# 08 · Complexity cheat sheet and gotchas

**How this notebook works**
- Run the setup cell below first. Then go top to bottom, running each cell with **Shift+Enter**.
- Gray **example** cells: run them, then change something and run again.
- **Try it** cells: your scratch space for each section.
- **Predict the output**: write your guess in the comment *before* running.
- **Exercises** at the bottom: write your code, run the cell, then run the `check(...)` cell under it.

In [ ]:
# ▶ Run this cell first (Shift+Enter). It loads the test helper and the same imports LeetCode gives you.
import sys, pathlib
sys.path.insert(0, str(next(p for p in (pathlib.Path.cwd() / "_primer", pathlib.Path.cwd() / "python-primer" / "_primer") if p.exists())))
from check import check, summary, lesson
lesson("ex08_gotchas")

import bisect, heapq, math, random
from collections import Counter, OrderedDict, defaultdict, deque
from functools import cache, lru_cache
from itertools import accumulate, combinations, groupby, pairwise, permutations, product
from typing import List, Optional

**Warm-up (5 min):** before reading, close everything and write down every trap you remember from lessons 01–07. Then check them against your *My traps* notes in those notebooks.

When an interviewer asks "what's the complexity?", hidden costs of Python operations are where answers go wrong. Learn this table.

## Cost of common operations

| Operation | Cost | Note |
|---|---|---|
| `lst.append(x)`, `lst.pop()`, `lst[i]`, `len(lst)` | O(1) | |
| `lst.pop(0)`, `lst.insert(0, x)`, `del lst[i]` | O(n) | shifts elements; use `deque` |
| `x in lst`, `lst.index(x)`, `lst.count(x)`, `lst.remove(x)` | O(n) | |
| `lst[a:b]`, `lst[:]`, `lst[::-1]`, `lst + other` | O(k) | creates a new list |
| `lst.sort()`, `sorted(lst)` | O(n log n) | |
| `min`, `max`, `sum`, `reversed` iteration | O(n) | |
| `d[k]`, `d[k] = v`, `k in d`, `del d[k]` | O(1) avg | same for sets |
| `Counter(lst)`, `set(lst)`, `dict(...)` | O(n) | building costs n |
| `a \| b`, `a & b` on sets | O(len(a) + len(b)) / O(min) | |
| `deque.append/appendleft/pop/popleft` | O(1) | |
| `deque[i]` in the middle | O(n) | |
| `heappush`, `heappop` | O(log n) | |
| `heapify` | O(n) | not n log n |
| `bisect_left/right` | O(log n) | |
| `insort` | O(n) | the insert shifts |
| `"".join(parts)` | O(total length) | |
| `s += c` in a loop | up to O(n²) total | build a list and join |
| `s[a:b]`, `s.lower()`, `s.split()`, `s[::-1]` | O(n) | strings are copied |
| `t in s` (substring) | O(len(s) · len(t)) worst | |
| Recursion | O(depth) stack space | count it in space complexity |

In [ ]:
import time
from collections import deque

n = 30_000
lst = list(range(n))
t = time.perf_counter()
while lst:
    lst.pop(0)                    # O(n) each -> O(n^2) total
print(f"list.pop(0)     x {n}: {time.perf_counter() - t:.3f}s")

dq = deque(range(n))
t = time.perf_counter()
while dq:
    dq.popleft()                  # O(1) each -> O(n) total
print(f"deque.popleft() x {n}: {time.perf_counter() - t:.3f}s")

big_list, big_set = list(range(n)), set(range(n))
t = time.perf_counter()
for _ in range(200):
    (n - 1) in big_list           # O(n) scan
print(f"200 x `in list`: {time.perf_counter() - t:.3f}s")
t = time.perf_counter()
for _ in range(200):
    (n - 1) in big_set            # O(1) hash lookup
print(f"200 x `in set`:  {time.perf_counter() - t:.5f}s")

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Gotchas, all in one place

1. `-7 // 2 == -4` and `-7 % 3 == 2`. Use `int(a / b)` for truncation.
2. `[[0] * c] * r` shares rows. Use `[[0] * c for _ in range(r)]`.
3. `nums = ...` inside a function doesn't modify the caller's list. Use `nums[:] = ...`.
4. `lst.sort()` returns `None`.
5. `res.append(path)` stores a reference. Use `path[:]`.
6. `if x:` is false for `0`. Use `if x is not None:` when 0 is valid.
7. `{}` is an empty dict. `set()` is an empty set.
8. Reading a missing `defaultdict` key inserts it.
9. Lists can't be dict keys or set members. Convert to a tuple.
10. Reassigning an outer variable in a nested function needs `nonlocal`.
11. `def f(acc=[])` shares one list across calls.
12. Recursion depth ~1000 by default.
13. Heap tuples with equal priorities compare the next element; add an index tiebreaker.
14. `heapq` is min-only; negate for max.
15. Ints never overflow; enforce 32-bit limits yourself when the problem asks.
16. Negative ints have infinite leading 1 bits; mask with `0xFFFFFFFF` for 32-bit bit problems.
17. Modifying a list, dict, or set while iterating over it. Iterate over a copy.
18. `s.split(" ")` keeps empty strings; `s.split()` doesn't.
19. Shadowing built-ins: naming a variable `list`, `dict`, `sum`, `min`, `max`, `id`, `str` breaks those functions for the rest of the function. Use `lst`, `total`, `lo`/`hi`.
20. `is` checks identity, `==` checks value. Use `is` only for `None`.

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Final self-test

Answer all of these out loud, then check. If you miss more than three, reread the lesson it came from.

1. What's the time complexity of this, and why?
   ```python
   while nums:
       x = nums.pop(0)
   ```
2. What's wrong here?
   ```python
   def dfs(i, path):
       if i == n:
           res.append(path)
       ...
   ```
3. Output? `print(sorted([3, 1, 2])[::-1][0])`
4. You need the 3 smallest of 10⁶ numbers. What's the cleanest one-liner and its complexity?
5. Why does `visited.add([r, c])` fail, and what's the fix?
6. Output?
   ```python
   a = [1, 2, 3]
   for x in a[:]:
       if x == 2:
           a.remove(x)
   print(a)
   ```

<details><summary>Answers</summary>

1. O(n²): each `pop(0)` is O(n). Use a deque and `popleft()`, or iterate by index.
2. It appends a reference to `path`; append `path[:]` (and make sure path is undone after recursing).
3. `3`
4. `heapq.nsmallest(3, nums)`, O(n log 3) which is O(n). Sorting would be O(n log n).
5. Lists aren't hashable. Use a tuple: `visited.add((r, c))`.
6. `[1, 3]` (safe, because it iterates over a copy)
</details>

---
# Exercises: fix the bugs

Every function below is already written and **broken**. For each one: delete the `raise NotImplementedError` line, run the cell and its `check(...)`, read the error, then fix the bug with the smallest change and add a one-line comment saying what was wrong.

**Given:** LeetCode defines `Node` for you. Run this cell, don't change it.

In [ ]:
class Node:
    def __init__(self, val, next=None):
        self.val = val
        self.next = next

### 1. `remove_zeros`

Remove all zeros from nums in place, keeping order.

```
nums = [0, 1, 0, 3, 12]
remove_zeros(nums)
nums should now be [1, 3, 12]
```

In [ ]:
def remove_zeros(nums: list) -> None:
    raise NotImplementedError
    nums = [x for x in nums if x != 0]

In [ ]:
check("remove_zeros")

### 2. `all_paths`

Every binary string of length n as a list of ints, e.g. n=2 -> [[0,0],[0,1],[1,0],[1,1]].

```
all_paths(2)  ->  [[0, 0], [0, 1], [1, 0], [1, 1]]
```

In [ ]:
def all_paths(n: int) -> list:
    raise NotImplementedError
    res, path = [], []

    def go():
        if len(path) == n:
            res.append(path)
            return
        for bit in (0, 1):
            path.append(bit)
            go()
            path.pop()

    go()
    return res

In [ ]:
check("all_paths")

### 3. `count_nodes_bfs`

Number of nodes reachable from start (including start).

```
count_nodes_bfs({1: [2, 3], 2: [4], 3: [4], 4: []}, 1)
  ->  4
```

In [ ]:
def count_nodes_bfs(graph: dict, start) -> int:
    raise NotImplementedError
    seen = set()
    q = deque([start])
    count = 0
    while q:
        node = q.popleft()
        count += 1
        seen.add(node)
        for nxt in graph[node]:
            if nxt not in seen:
                q.append(nxt)
    return count

In [ ]:
check("count_nodes_bfs")

### 4. `tree_sum`

Sum of node values in a binary tree given as nested tuples (val, left, right) or None.

```
tree_sum((1, (2, None, None), (3, (4, None, None), None)))  ->  10
tree_sum(None)                                              ->  0
```

In [ ]:
def tree_sum(root) -> int:
    raise NotImplementedError
    total = 0

    def dfs(node):
        if node is None:
            return
        val, left, right = node
        total += val
        dfs(left)
        dfs(right)

    dfs(root)
    return total

In [ ]:
check("tree_sum")

### 5. `merge_k_linked`

Merge sorted linked lists (Node objects) and return the values as a Python list.

```
merge_k_linked([a, b, None])  ->  [1, 1, 3, 4]
```

In [ ]:
def merge_k_linked(heads: list) -> list:
    raise NotImplementedError
    h = []
    for node in heads:
        if node:
            heapq.heappush(h, (node.val, node))
    out = []
    while h:
        val, node = heapq.heappop(h)
        out.append(val)
        if node.next:
            heapq.heappush(h, (node.next.val, node.next))
    return out

In [ ]:
check("merge_k_linked")

### 6. `window_max_count`

How many windows of size k have their maximum at the window's last position? (A deliberately simple brute force; the bug is not about efficiency.)

```
window_max_count([1, 3, 2, 5, 4, 6], 2)  ->  3
```

In [ ]:
def window_max_count(nums: list, k: int) -> int:
    raise NotImplementedError
    max = 0
    for i in range(k - 1, len(nums)):
        window = nums[i - k + 1:i + 1]
        if window[-1] == max(window):
            max += 1
    return max

In [ ]:
check("window_max_count")

### 7. `make_board`

n x n board of '.' strings where each row can be edited independently.

```
b = make_board(3)
b[0][0] = "Q"
only row 0 should change: [row[0] for row in b] == ["Q", ".", "."]
```

In [ ]:
def make_board(n: int) -> list:
    raise NotImplementedError
    return [["."] * n] * n

In [ ]:
check("make_board")

### 8. `contains`

Does nums contain target? (Written the long way on purpose.)

```
contains([5, 6], 6)  ->  True
contains([5, 6], 5)  ->  True
contains([5, 6], 9)  ->  False
```

In [ ]:
def contains(nums: list, target) -> bool:
    raise NotImplementedError
    idx = None
    for i, x in enumerate(nums):
        if x == target:
            idx = i
            break
    return True if idx else False

In [ ]:
check("contains")

### 9. `safe_lookup_count`

For each query, how many times it appears in words. Must not add queries to the counts.

```
safe_lookup_count(['a', 'b', 'a'], ['a', 'z'])  ->  [2, 0]
```

In [ ]:
def safe_lookup_count(words: list, queries: list) -> list:
    raise NotImplementedError
    from collections import defaultdict
    counts = defaultdict(int)
    for w in words:
        counts[w] += 1
    answer = [counts[q] for q in queries]
    assert len(counts) == len(set(words)), "queries leaked into counts"
    return answer

In [ ]:
check("safe_lookup_count")

---
## Wrap-up

In [ ]:
summary()

## My traps

*Double-click to edit.* Write the 3–5 things from this lesson that surprised you or that you got wrong, in your own words. You'll use these in the next warm-up and the exit test.

1. 
2. 
3. 

**Done?** All `ok` in the summary → commit and sync → tell Claude **"lesson 08 done."**